### Importing libraries

In [1]:
import os
import json
from dotenv import load_dotenv

from chromadb import PersistentClient

from typing import TypedDict

import ollama
from tavily import TavilyClient
from google import genai

from langgraph.graph import StateGraph, END

### Setting up API keys and loading ChromaDB :

In [2]:
load_dotenv()

gemini_api_key = os.getenv("gemini_api_key")
tavily_api_key = os.getenv("tavily_api_key")


client = PersistentClient(path="../knowledge_base/chroma_db")

collection = client.get_collection(name="ai_assistant")

### Warming up the embedding model:

In [3]:
def warm_up_embedding_model():
    print("Warming up embeddinggemma...")
    ollama.embed(model="embeddinggemma", input="warmup")
    print("Embedding model ready.")

warm_up_embedding_model()

Warming up embeddinggemma...
Embedding model ready.


### Setting up Gemini:

In [4]:
def getResponseFromLLM(system_prompt : str, user_prompt : str, model_temp : float, format : str = "json"):

    client = genai.Client(api_key=gemini_api_key)
    response_format = "application/json" if format == "json" else "text/plain"

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=user_prompt,
        config=genai.types.GenerateContentConfig(
                system_instruction=system_prompt,
                response_mime_type=response_format,
                temperature= model_temp
            )
        )
    
    return response

### Setting up the router:

In [5]:
ROUTER_SYSTEM_INSTRUCTION = """
You are an expert financial and legal intent classifier for a Retrieval-Augmented Generation (RAG) system. 
Your job is to categorize a user's query into exactly one of the following three categories.

### Categories:
1. "ifrs": 
   - Use this for questions regarding "International Financial Reporting Standards" (IAS/IFRS).
   - Keywords: IFRS, IAS, International accounting, consolidation (international context).

2. "tax_code":
   - Use this for questions regarding the **Tunisian** tax system.
   - Includes: Code de l'IRPP et de l'IS, TVA (VAT), fiscal procedures, registration duties, and local finance laws in Tunisia.
   - Any vague question about "tax" or "fisc" implies Tunisia unless stated otherwise.

3. "accounting_standards":
   - Use this for questions regarding **Tunisian** local accounting standards.
   - Includes: The "Système Comptable des Entreprises" (SCE), local chart of accounts (NCT / Normes Comptables Tunisiennes).

4. "web_search": REQUIRES live internet data.
   - Includes: Current exchange rates, 2025 news, specific recent Tunisian political events, or specific data from the current year.

5. "general_knowledge": The LLM can answer this immediately. 
   - Includes: Greetings, general definitions ("What is an asset?"), generic advice, or simple explanations of concepts.

### Output Format:
You must output ONLY a JSON object with a single key "category".
Example: {"category": "tax_code"}
"""

def route_query(user_query : str):
    """
    Routes the user query using Gemini Flash to decide which database to search.
    """
    try:
        response = getResponseFromLLM(ROUTER_SYSTEM_INSTRUCTION,user_query,0.0)
        if response.text is None:
            raise ValueError("LLM response is empty")
        
        result = json.loads(response.text)
        return result.get("category", "general_knowledge")

    except Exception as e:
        print(f"Router Error: {e}")
        return "general_knowledge"

### Setting up the retrieval function :

In [6]:
def retrieve_context(query: str, category, n_results: int = 5):
    """
    Retrieves relevant chunks from ChromaDB filtered by category.
    """
    query_embedding = ollama.embed(
        model="embeddinggemma",
        input=query
    )["embeddings"][0]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        where={"category": {"$eq": category}}
    )

    documents = results.get("documents", [[]])
    context_list = documents[0] if documents and len(documents) > 0 else []
    if not context_list or not isinstance(context_list, list):
        return "No local documents found."
    return "\n\n".join(context_list)

### Setting up web search :

In [7]:
tavily = TavilyClient(api_key=tavily_api_key)

def search_web(query: str):
    """
    Performs a search using Tavily and returns a clean context string.
    """
    try:
        response = tavily.search(
            query=query,
            search_depth="advanced",
            max_results=5
        )
        context = ""
        for result in response['results']:
            context += f"Source: {result['url']}\nContent: {result['content']}\n\n"
            
        return context if context else "No web results found."
    except Exception as e:
        print(f"Tavily Error: {e}")
        return "Web search failed."

### Defining the graph state :

In [8]:
class GraphState(TypedDict):
    query: str
    category: str
    context: str
    answer: str
    is_valid: bool

### Defining the graph nodes :

In [ ]:
def router_node(state: GraphState):
    """
    Named function allows for better error logging and type checking.
    """
    user_query = state["query"] 
    category = route_query(user_query)
    return {"category": category}

def retrieval_node(state: GraphState):
    print(f"--- NODE: RETRIEVING FROM {state["category"]} ---")
    context = retrieve_context(state["query"], state["category"], 5)
    
    return {"context": context}

def web_search_node(state: GraphState):
    print("--- NODE: WEB SEARCHING (TAVILY) ---")
    context = search_web(state["query"])
    return {"context": context}

VALIDATOR_SYSTEM_INSTRUCTION = """
You are a "Context Judge". Your sole task is to determine if the provided CONTEXT contains enough relevant information to accurately answer the USER QUERY.

Rules:
1. If the context is relevant and provides an answer (even partially), return {"is_valid": true}.
2. If the context is completely unrelated, nonsensical, or states no information is found, return {"is_valid": false}.
3. Do NOT try to answer the query itself. Just judge the relationship between the query and the context.

Output ONLY JSON: {"is_valid": boolean}
"""

def validate_node(state: GraphState):
    print("--- NODE: VALIDATING CONTEXT RELAVANCE ---")
    if state["category"] == "general_knowledge":
        return {"is_valid": True}

    user_input = f"USER QUERY: {state["query"]}\n\nRETRIEVED CONTEXT: {state["context"]}"
    
    try:
        response = getResponseFromLLM(VALIDATOR_SYSTEM_INSTRUCTION, user_input, 0.0)
        if response.text is None:
            raise ValueError("LLM Response is empty")
        result = json.loads(response.text)
        
        is_valid = result.get("is_valid", False)
        return {"is_valid": is_valid}
    except Exception as e:
        print(f"Validation Error: {e}")
        return {"is_valid": False}

def generate_answer_node(state: GraphState):
    print("--- NODE: GENERATING ANSWER NODE ---")

    context = state.get("context", "")
    query = state.get("query", "")
    category = state.get("category", "general_knowledge")

    if not context or category == "general_knowledge":
        expert_prompt = """
            You are a Senior Financial Advisor. 
            Detect the language of the user's query and respond in that same language (e.g., English or French).
            Provide a professional, friendly, and concise response. 
            Since this is a general query, you do not need to cite specific Tunisian articles 
            unless they are part of your general knowledge. 
            Do not mention the language detection process in your response.
            """
        user_msg = f"QUESTION: {query}"
    else:
        expert_prompt = """
        You are a Senior Financial Advisor and Legal Expert in Tunisia. 
        Detect the language of the user's query and provide a high-quality, professional response 
        in that same language (e.g., English or French), based strictly on the provided context.

        ### Formatting Rules:
        1. **Language**: Respond exclusively in the language used by the user.
        2. **Structure**: Provide your answer as a single, well-structured, and concise paragraph.
        3. **Citations**: 
            - For 'tax_code', integrate citations of specific Articles (e.g., Code de l'IRPP) directly into the flow of the text.
            - For 'ifrs', use standard terminology (e.g., IFRS 16) within the prose.
            - For 'web_search', end the paragraph with a source sentence in the user's language:
                * If English: "*Source: Information retrieved from recent online financial data.*"
                * If French: "*Source: Informations extraites de données financières en ligne récentes.*"
        4. **Tone**: Maintain a formal, authoritative, and advisory narrative style.

        ### Goal:
        Deliver a direct answer in a narrative format. If the context is insufficient, 
        state exactly what is missing regarding Tunisian regulations within that same 
        paragraph, using the user's language.
        """
        user_msg = f"CONTEXT: {context}\n\nQUESTION: {query}"
    
    response = getResponseFromLLM(
        system_prompt=expert_prompt, 
        user_prompt=user_msg, 
        model_temp=0.5, 
        format="text"
    )
    
    return {"answer": response.text}

### Building the pipeline:

In [ ]:
def decide_next_node(state: GraphState):
    if state["category"] == "web_search":
        return "web_search"
    elif state["category"] == "general_knowledge":
        return "generate"
    else:
        return "retrieve"

def post_val_routing(state: GraphState):
    """
    Determines if we go to generation or try a web search fallback.
    """
    if state["is_valid"]:
        return "generate"
    else:
        print("--- VALIDATION FAILED: FALLING BACK TO WEB SEARCH ---")
        return "web_search"


workflow = StateGraph(GraphState)

workflow.add_node("router", router_node)
workflow.add_node("retrieve", retrieval_node)
workflow.add_node("validate", validate_node)
workflow.add_node("web_search", web_search_node)
workflow.add_node("generate", generate_answer_node)

workflow.set_entry_point("router")

workflow.add_conditional_edges(
    "router",
    decide_next_node,
    {
        "web_search": "web_search",
        "generate": "generate",
        "retrieve": "retrieve"
    }
)

workflow.add_edge("retrieve", "validate")

workflow.add_conditional_edges(
    "validate",
    post_val_routing,
    {
        "generate": "generate",
        "web_search": "web_search"
    }
)

workflow.add_edge("web_search", "generate")
workflow.add_edge("generate", END)

app = workflow.compile()

### Testing the workflow:

In [13]:
test_input = {"query": "Quel est le taux d'echange de dinar tunisien par rapport au euro?"}

try:
    final_state = app.invoke(test_input) # type: ignore
    print("Answer:", final_state["answer"])
except Exception as e:
    print(f"Execution Error: {e}")

--- NODE: WEB SEARCHING (TAVILY) ---
--- NODE: GENERATING ANSWER NODE ---
Answer: Actuellement, le taux de change du dinar tunisien par rapport à l'euro est d'environ 1 TND pour 0,296329 EUR, selon le taux interbancaire. Il est important de noter que ces taux sont indicatifs et peuvent varier légèrement entre les différentes plateformes et institutions financières. Source: Informations extraites de données financières en ligne récentes.
